# Late Delivery Root-Cause Analysis (Olist Brazilian E-commerce)
**Tools:** Python, SQL (SQLite), Tableau
**Goal:** find what drives late deliveries (delivered after the estimated date) by region, seller, product category, and fulfillment stage.

Every number in your resume must come from the **"Resume metrics"** cell at the end of this notebook.
Run the cells top to bottom.

## 1. Load the data
Use **Option A** (download in Colab) *or* **Option B** (upload the zip you downloaded to your PC). Run only one.

In [ ]:
# OPTION A: download directly in Colab with kagglehub
import kagglehub
DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Path to dataset files:", DATA_DIR)

In [ ]:
# OPTION B: upload the zip you downloaded to your PC (skip if Option A worked)
# from google.colab import files
# import zipfile, os
# up = files.upload()                      # choose the archive.zip from your PC
# zname = list(up.keys())[0]
# DATA_DIR = "/content/olist"
# os.makedirs(DATA_DIR, exist_ok=True)
# zipfile.ZipFile(zname).extractall(DATA_DIR)
# print(os.listdir(DATA_DIR))

In [ ]:
import os, glob, sqlite3
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    assert hits, f"{name} not found under {DATA_DIR}"
    return hits[0]

files = {
    "orders":       "olist_orders_dataset.csv",
    "items":        "olist_order_items_dataset.csv",
    "customers":    "olist_customers_dataset.csv",
    "sellers":      "olist_sellers_dataset.csv",
    "products":     "olist_products_dataset.csv",
    "reviews":      "olist_order_reviews_dataset.csv",
    "translation":  "product_category_name_translation.csv",
}
dfs = {k: pd.read_csv(find(v)) for k, v in files.items()}
for k, d in dfs.items():
    print(f"{k:12s} {d.shape}")

## 2. Load into a SQL database
SQLite ships with Colab, so no setup is needed. Dates are kept as `YYYY-MM-DD HH:MM:SS` text so date functions work.

In [ ]:
conn = sqlite3.connect("olist.db")
for k, d in dfs.items():
    d.to_sql(k, conn, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT name FROM sqlite_master WHERE type='table'")

## 3. Data quality checks
Check these before analyzing. Only delivered orders with a delivery date can be classified as late or on time.

In [ ]:
print(q('''
SELECT order_status, COUNT(*) AS orders,
       SUM(order_delivered_customer_date IS NULL) AS missing_delivery_date
FROM orders GROUP BY order_status ORDER BY orders DESC
'''))
print(q('''
SELECT MIN(order_purchase_timestamp) AS first_order, MAX(order_purchase_timestamp) AS last_order
FROM orders
'''))

## 4. Build one analysis table (one row per delivered order)
Rules used (state them in your README):
- Only `order_status = 'delivered'` with a non-null delivery date.
- **Late** = delivered after the estimated delivery date.
- An order can have several items/sellers. The seller, product and category are taken from **item #1**.
- **Seller stage** = purchase to carrier handoff. **Carrier stage** = handoff to customer.
- **Late handoff** = seller handed to the carrier after the `shipping_limit_date`.

In [ ]:
conn.executescript('''
DROP TABLE IF EXISTS master;
CREATE TABLE master AS
WITH first_item AS (
    SELECT order_id, seller_id, product_id, shipping_limit_date
    FROM items WHERE order_item_id = 1
),
freight AS (
    SELECT order_id, SUM(freight_value) AS freight_value, SUM(price) AS price
    FROM items GROUP BY order_id
),
rev AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM reviews GROUP BY order_id
)
SELECT
    o.order_id,
    o.order_purchase_timestamp                    AS purchased,
    o.order_delivered_carrier_date                AS carrier_date,
    o.order_delivered_customer_date               AS delivered,
    o.order_estimated_delivery_date               AS estimated,
    c.customer_state,
    s.seller_id,
    s.seller_state,
    COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
    f.price, f.freight_value,
    r.review_score,
    CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END AS is_late,
    julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date)       AS days_vs_estimate,
    julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)            AS total_days,
    julianday(o.order_delivered_carrier_date)  - julianday(o.order_purchase_timestamp)            AS seller_stage_days,
    julianday(o.order_delivered_customer_date) - julianday(o.order_delivered_carrier_date)        AS carrier_stage_days,
    CASE WHEN o.order_delivered_carrier_date > fi.shipping_limit_date THEN 1 ELSE 0 END           AS late_handoff,
    CASE WHEN s.seller_state = c.customer_state THEN 1 ELSE 0 END                                 AS same_state
FROM orders o
JOIN customers c    ON c.customer_id = o.customer_id
JOIN first_item fi  ON fi.order_id   = o.order_id
JOIN sellers s      ON s.seller_id   = fi.seller_id
LEFT JOIN products p ON p.product_id = fi.product_id
LEFT JOIN translation t ON t.product_category_name = p.product_category_name
LEFT JOIN freight f ON f.order_id = o.order_id
LEFT JOIN rev r     ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_delivered_carrier_date  IS NOT NULL;
''')
q("SELECT COUNT(*) AS delivered_orders, SUM(is_late) AS late_orders FROM master")

## 5. Analysis
### Q1. How big is the problem?

In [ ]:
overall = q('''
SELECT COUNT(*) AS orders,
       SUM(is_late) AS late_orders,
       ROUND(100.0*AVG(is_late), 2) AS late_rate_pct,
       ROUND(AVG(CASE WHEN is_late=1 THEN days_vs_estimate END), 1) AS avg_days_late_when_late,
       ROUND(AVG(total_days), 1) AS avg_total_delivery_days
FROM master
''')
overall

### Q2. Which customer states have the worst late rate? (min. 500 orders)

In [ ]:
by_state = q('''
SELECT customer_state, COUNT(*) AS orders, SUM(is_late) AS late_orders,
       ROUND(100.0*AVG(is_late), 2) AS late_rate_pct,
       ROUND(AVG(CASE WHEN is_late=1 THEN days_vs_estimate END), 1) AS avg_days_late
FROM master GROUP BY customer_state
HAVING COUNT(*) >= 500
ORDER BY late_rate_pct DESC
''')
display(by_state)
by_state.sort_values("late_rate_pct").plot.barh(x="customer_state", y="late_rate_pct", legend=False, figsize=(7,6))
plt.xlabel("Late rate (%)"); plt.title("Late delivery rate by customer state"); plt.show()

### Q3. Which product categories are worst? (min. 300 orders)

In [ ]:
by_cat = q('''
SELECT category, COUNT(*) AS orders,
       ROUND(100.0*AVG(is_late), 2) AS late_rate_pct,
       ROUND(AVG(freight_value), 2) AS avg_freight
FROM master GROUP BY category
HAVING COUNT(*) >= 300
ORDER BY late_rate_pct DESC
''')
by_cat.head(10)

### Q4. Is it a seller problem? Seller concentration
Window functions: `NTILE` splits sellers into deciles by late-order count. Most sellers have few orders, so also check Q4b.

In [ ]:
concentration = q('''
WITH s AS (
    SELECT seller_id, COUNT(*) AS orders, SUM(is_late) AS late_orders
    FROM master GROUP BY seller_id
),
r AS (
    SELECT seller_id, orders, late_orders,
           NTILE(10) OVER (ORDER BY late_orders DESC, seller_id) AS decile
    FROM s
)
SELECT decile, COUNT(*) AS sellers, SUM(orders) AS orders, SUM(late_orders) AS late_orders,
       ROUND(100.0*SUM(late_orders)/(SELECT SUM(late_orders) FROM s), 1) AS pct_of_all_late_orders
FROM r GROUP BY decile ORDER BY decile
''')
concentration

In [ ]:
# Q4b: sellers with enough volume to judge fairly (>= 30 orders), ranked by late rate
bad_sellers = q('''
SELECT seller_id, seller_state, COUNT(*) AS orders, SUM(is_late) AS late_orders,
       ROUND(100.0*AVG(is_late), 1) AS late_rate_pct,
       RANK() OVER (ORDER BY AVG(is_late) DESC) AS rnk
FROM master GROUP BY seller_id, seller_state
HAVING COUNT(*) >= 30
ORDER BY late_rate_pct DESC LIMIT 15
''')
bad_sellers

### Q5. Where in the journey does the delay happen? Seller stage vs. carrier stage

In [ ]:
stage = q('''
SELECT is_late,
       COUNT(*) AS orders,
       ROUND(AVG(seller_stage_days), 1)  AS avg_days_purchase_to_handoff,
       ROUND(AVG(carrier_stage_days), 1) AS avg_days_handoff_to_customer
FROM master GROUP BY is_late
''')
display(stage)

handoff = q('''
SELECT late_handoff, COUNT(*) AS orders,
       ROUND(100.0*AVG(is_late), 2) AS late_rate_pct
FROM master GROUP BY late_handoff
''')
display(handoff)

### Q6. Does the seller being in the same state as the customer matter?

In [ ]:
q('''
SELECT same_state, COUNT(*) AS orders,
       ROUND(100.0*AVG(is_late), 2) AS late_rate_pct,
       ROUND(AVG(total_days), 1) AS avg_total_days
FROM master GROUP BY same_state
''')

### Q7. What does lateness do to customer reviews?

In [ ]:
reviews_impact = q('''
SELECT is_late, COUNT(review_score) AS reviewed_orders,
       ROUND(AVG(review_score), 2) AS avg_review_score,
       ROUND(100.0*AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0 END), 1) AS pct_1_or_2_stars
FROM master GROUP BY is_late
''')
reviews_impact

### Q8. Monthly trend (window function `LAG`, months with 100+ orders)

In [ ]:
trend = q('''
WITH m AS (
    SELECT strftime('%Y-%m', purchased) AS ym, COUNT(*) AS orders,
           ROUND(100.0*AVG(is_late), 2) AS late_rate_pct
    FROM master GROUP BY ym HAVING COUNT(*) >= 100
)
SELECT ym, orders, late_rate_pct,
       ROUND(late_rate_pct - LAG(late_rate_pct) OVER (ORDER BY ym), 2) AS change_vs_prev_month
FROM m ORDER BY ym
''')
display(trend)
trend.plot(x="ym", y="late_rate_pct", figsize=(9,3.5), marker="o", legend=False)
plt.ylabel("Late rate (%)"); plt.title("Monthly late-delivery rate"); plt.xticks(rotation=60); plt.show()

## 6. Resume metrics
Copy numbers **from this output only**.

In [ ]:
o = overall.iloc[0]
top_states = by_state.head(3)
top_decile_share = concentration.iloc[0]["pct_of_all_late_orders"]
lh = handoff.set_index("late_handoff")["late_rate_pct"]
same = q("SELECT same_state, ROUND(100.0*AVG(is_late),2) AS r FROM master GROUP BY same_state").set_index("same_state")["r"]
rv = reviews_impact.set_index("is_late")["avg_review_score"]

print(f"Orders analyzed (delivered, with dates): {int(o.orders):,}")
print(f"Overall late rate: {o.late_rate_pct}%  | avg delay when late: {o.avg_days_late_when_late} days")
print("Worst 3 states (>=500 orders):")
print(top_states[["customer_state","orders","late_rate_pct"]].to_string(index=False))
print(f"Top 10% of sellers (by late-order count) account for {top_decile_share}% of all late orders")
print(f"Late rate if seller handed off late: {lh.get(1)}%  vs on time: {lh.get(0)}%")
print(f"Late rate cross-state: {same.get(0)}%  vs same-state: {same.get(1)}%")
print(f"Avg review score: late {rv.get(1)} vs on-time {rv.get(0)}")

## 7. Export for Tableau

In [ ]:
os.makedirs("tableau_exports", exist_ok=True)
q("SELECT * FROM master").to_csv("tableau_exports/master_orders.csv", index=False)
by_state.to_csv("tableau_exports/late_by_state.csv", index=False)
by_cat.to_csv("tableau_exports/late_by_category.csv", index=False)
trend.to_csv("tableau_exports/monthly_trend.csv", index=False)
bad_sellers.to_csv("tableau_exports/worst_sellers.csv", index=False)

import shutil
shutil.make_archive("tableau_exports", "zip", "tableau_exports")
try:
    from google.colab import files
    files.download("tableau_exports.zip")
except ImportError:
    print("Saved tableau_exports.zip locally")

## 8. Tableau dashboard
In Tableau, connect to `master_orders.csv` and build one dashboard with:
1. KPI tiles: late rate, avg days late, avg review score (late vs on-time)
2. Filled map or bar by `customer_state`: late rate
3. Bar by `category`: late rate (filter orders >= 300)
4. Line chart by month: late rate
5. Table of the worst sellers (`worst_sellers.csv`)
6. Filters: state, category, month

Publish it to Tableau Public and add the link to your resume and README.

## MySQL note
Queries use SQLite date functions (`julianday`, `strftime`). For MySQL 8, replace them with `DATEDIFF` / `TIMESTAMPDIFF` and `DATE_FORMAT`. CTEs, `NTILE`, `RANK` and `LAG` work as-is. Say "SQL (SQLite/MySQL)" on your resume only if you do run it in MySQL.